<a href="https://colab.research.google.com/github/Md-Golam-Raiyhan/INSE-6450-Milestone-1/blob/main/generate_synthetic_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os
import numpy as np
import pandas as pd

RNG_SEED = 42

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def clamp_nonnegative(x: float) -> float:
    return max(0.0, x)

def main(
    out_dir: str = "data",
    weeks: int = 6,
) -> None:
    """
    Generates synthetic datasets for:
      - pantry.csv
      - consumption_log.csv
      - recipes.csv
      - user_profile.csv
    """
    np.random.seed(RNG_SEED)
    ensure_dir(out_dir)

    # -------------------------
    # 1) Define item catalog
    # -------------------------
    # avg_weekly_use is the typical weekly usage per household.
    items = [
        # item_id, item_name, category, unit, essential, start_qty, avg_weekly_use
        (1,  "Rice",     "Grain",      "kg",    1, 5.0, 0.9),
        (2,  "Eggs",     "Dairy",      "piece", 1, 12,  6),
        (3,  "Milk",     "Dairy",      "liter", 1, 2.0, 1.2),
        (4,  "Chicken",  "Meat",       "kg",    0, 1.5, 0.6),
        (5,  "Onion",    "Vegetable",  "piece", 1, 6,   3),
        (6,  "Tomato",   "Vegetable",  "piece", 0, 6,   3),
        (7,  "Potato",   "Vegetable",  "piece", 1, 10,  5),
        (8,  "Bread",    "Bakery",     "piece", 1, 2,   1),
        (9,  "Yogurt",   "Dairy",      "piece", 0, 6,   3),
        (10, "Lentils",  "Grain",      "kg",    1, 2.0, 0.4),
        (11, "Pasta",    "Grain",      "kg",    0, 2.0, 0.5),
        (12, "Apple",    "Fruit",      "piece", 0, 10,  5),
        (13, "Banana",   "Fruit",      "piece", 0, 10,  6),
        (14, "Spinach",  "Vegetable",  "piece", 0, 2,   1),
        (15, "Cheese",   "Dairy",      "kg",    0, 0.8, 0.2),
    ]

    items_df = pd.DataFrame(
        items,
        columns=["item_id", "item_name", "category", "unit", "essential", "start_quantity", "avg_weekly_use"]
    )

    # Pantry is your "current state" at week 1 (start)
    pantry_df = items_df[["item_id", "item_name", "category", "unit", "essential"]].copy()
    pantry_df["quantity"] = items_df["start_quantity"]

    # Save pantry.csv
    pantry_path = os.path.join(out_dir, "pantry.csv")
    pantry_df.to_csv(pantry_path, index=False)

    # -------------------------
    # 2) Generate weekly consumption log
    # -------------------------
    # We simulate weekly usage around avg_weekly_use with some noise.
    # For count-like items (piece), we use Poisson; for continuous (kg/liter), Normal.
    consumption_rows = []

    for week in range(1, weeks + 1):
        for _, row in items_df.iterrows():
            unit = row["unit"]
            mu = float(row["avg_weekly_use"])

            if unit == "piece":
                used = np.random.poisson(lam=max(mu, 0.1))
                used = int(used)
            else:
                # Normal noise around mu, then clamp nonnegative and round to 2 decimals
                used = np.random.normal(loc=mu, scale=max(0.15 * mu, 0.05))
                used = round(clamp_nonnegative(used), 2)

            consumption_rows.append({
                "week": week,
                "item_id": int(row["item_id"]),
                "quantity_used": used,
                "unit": unit
            })

    consumption_df = pd.DataFrame(consumption_rows)

    # Optional: add a tiny amount of missingness (to demonstrate cleaning in Milestone 1)
    # Comment this out if you want perfectly clean data.
    miss_mask = np.random.rand(len(consumption_df)) < 0.01  # 1% missing
    consumption_df.loc[miss_mask, "quantity_used"] = np.nan

    consumption_path = os.path.join(out_dir, "consumption_log.csv")
    consumption_df.to_csv(consumption_path, index=False)

    # -------------------------
    # 3) Create a small recipe database
    # -------------------------
    # Keep it small (10–15) for the project.
    recipes = [
        (1, "Vegetable Fried Rice", "Rice,Onion,Tomato,Spinach", 25, "low",    "veg",     "none"),
        (2, "Chicken Fried Rice",   "Rice,Chicken,Eggs,Onion",   30, "medium", "non-veg", "egg"),
        (3, "Egg Curry",            "Eggs,Onion,Tomato",         20, "low",    "non-veg", "egg"),
        (4, "Lentil Soup",          "Lentils,Onion,Tomato",      35, "low",    "veg",     "none"),
        (5, "Pasta with Tomato",    "Pasta,Tomato,Onion,Cheese", 25, "medium", "veg",     "milk"),
        (6, "Omelette",             "Eggs,Onion,Spinach",        15, "low",    "veg",     "egg"),
        (7, "Chicken & Potatoes",   "Chicken,Potato,Onion",      40, "medium", "non-veg", "none"),
        (8, "Fruit Bowl",           "Apple,Banana,Yogurt",       10, "low",    "veg",     "milk"),
        (9, "Milk Banana Smoothie", "Milk,Banana,Yogurt",        10, "low",    "veg",     "milk"),
        (10,"Rice & Lentils",       "Rice,Lentils,Onion",        30, "low",    "veg",     "none"),
    ]

    recipes_df = pd.DataFrame(
        recipes,
        columns=["recipe_id", "recipe_name", "ingredients", "cooking_time_min", "cost_level", "diet", "allergens"]
    )

    recipes_path = os.path.join(out_dir, "recipes.csv")
    recipes_df.to_csv(recipes_path, index=False)

    # -------------------------
    # 4) User profile / constraints
    # -------------------------
    user_profile_df = pd.DataFrame([{
        "user_id": 1,
        "diet": "non-veg",          # or "veg"
        "allergies": "egg",         # comma-separated: "egg,milk" or "none"
        "max_time_min": 30,
        "weekly_budget": "low"      # "low" or "medium"
    }])

    user_path = os.path.join(out_dir, "user_profile.csv")
    user_profile_df.to_csv(user_path, index=False)

    print("✅ Synthetic data generated:")
    print(f" - {pantry_path}")
    print(f" - {consumption_path}")
    print(f" - {recipes_path}")
    print(f" - {user_path}")

if __name__ == "__main__":
    main()


✅ Synthetic data generated:
 - data/pantry.csv
 - data/consumption_log.csv
 - data/recipes.csv
 - data/user_profile.csv


In [5]:
 from google.colab import files

files.download("data/pantry.csv")
files.download("data/consumption_log.csv")
files.download("data/recipes.csv")
files.download("data/user_profile.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>